# Práctica guiada — De formulario a experiencia de inferencia

En S5 ya construisteis un formulario Streamlit que recoge los 11 atributos y llama a un gateway basado en el contrato de S4. En S6 **no se rediseña el contrato ni se vuelve a implementar el formulario**. El objetivo es convertir esa pantalla funcional en una interfaz operable: mantiene el resultado al haber un rerun, hace visible lo que está ocurriendo y enseña la información justa para interpretar e investigar una predicción.

Trabajad en parejas. Completad los `TODO`, ejecutad las comprobaciones y conservad el notebook como evidencia para el taller. La implementación queda para la Clase 2.

## Entrega de la práctica

Al terminar debéis poder enseñar cuatro decisiones concretas:

1. qué permanece en la sesión, qué se cachea y qué no se retiene;
2. la máquina `idle → loading → success/error`;
3. un plano de pantalla con componentes Streamlit concretos;
4. mensajes seguros para confianza baja, lentitud y error.

No se entrega código de inferencia nuevo. S4 sigue siendo la única frontera de validación y predicción.

## 1. Comprobar el punto de partida de S5

Ejecutad la celda. Debe demostrar que partís del formulario y gateway de S5, no de una segunda implementación. Después anotad una limitación que observaríais al cambiar un widget o al enviar una segunda predicción.

In [1]:
from pathlib import Path

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'semana5/modules/05-streamlit-basic-model-ui').is_dir()
)
s5_app = repo_root / 'semana5/modules/05-streamlit-basic-model-ui/solutions/02-first-streamlit/app.py'
source = s5_app.read_text(encoding='utf-8')

checks = {
    'reutiliza st.form': 'st.form(' in source,
    'recoge valores antes de inferir': 'collect_values' in source,
    'aún no usa session_state': 'st.session_state' not in source,
}
print('app de S5:', s5_app.relative_to(repo_root).as_posix())
for label, passed in checks.items():
    print(f"{'✓' if passed else '✗'} {label}")
assert all(checks.values())

# Limitación observable de S5.
limitacion_s5 = (
    'El resultado de S5 solo existe dentro de `if submitted`: al tocar cualquier '
    'otro widget el script se relanza y la predicción desaparece. Además, '
    'run_configured_app() llama a gateway_factory() en cada rerun (recarga el '
    'bundle), no hay estado de carga, no se distingue una respuesta lenta de un '
    'fallo y los errores no ofrecen reintento ni request_id.'
)
print('Limitación observada:', limitacion_s5)

app de S5: semana5/modules/05-streamlit-basic-model-ui/solutions/02-first-streamlit/app.py
✓ reutiliza st.form
✓ recoge valores antes de inferir
✓ aún no usa session_state
Limitación observada: El resultado de S5 solo existe dentro de `if submitted`: al tocar cualquier otro widget el script se relanza y la predicción desaparece. Además, run_configured_app() llama a gateway_factory() en cada rerun (recarga el bundle), no hay estado de carga, no se distingue una respuesta lenta de un fallo y los errores no ofrecen reintento ni request_id.


## 2. Decidir dónde vive cada cosa

Un rerun vuelve a ejecutar el script. Una variable local desaparece; `st.session_state` conserva información de **esa sesión**; `st.cache_resource` reutiliza recursos como un gateway o modelo. Un recurso cacheado global puede compartirse entre usuarios, así que debe ser seguro para concurrencia.

Completad la tabla y justificad cada decisión. No guardéis secretos ni las 11 features en la telemetría. Los valores del último envío solo se conservan para permitir un reintento y no se muestran ni se registran.

| Clave o recurso | Dónde vive | ¿Sobrevive a un rerun? | ¿Se limpia al pulsar «Limpiar»? | Motivo |
| --- | --- | --- | --- | --- |
| `last_state` | `st.session_state`, tipo `UiState` (`phase`, `request_id`, `view`, `error`) | Sí | Sí: vuelve a `UiState(phase="idle")` | Es lo que la persona está mirando. Sin él, el resultado desaparece al tocar cualquier widget (limitación de S5). |
| `last_values` | `st.session_state`, tipo `dict[str, float]` | Sí | Sí: vuelve a `{}` | Solo sirve para «Reintentar» el último envío sin rellenar el formulario. No se muestra ni se registra, y desaparece con limpiar. |
| `telemetry` | `st.session_state`, tipo `Telemetry` | Sí | **No** | Es observabilidad de la sesión (contadores, errores por código, latencias, versiones). Limpiar la pantalla no puede borrar evidencia operativa. |
| gateway / bundle | `st.cache_resource` (memoria del proceso, compartido entre sesiones) | Sí, y entre sesiones | No. Solo se invalida al reintentar tras `artifact_unavailable` o reiniciando la app | Cargar el `.joblib` es caro e igual para todos. Debe ser de solo lectura en `predict` para ser seguro con concurrencia. |
| respuesta de una petición | Variable local en `controller.submit`; lo que se conserva es su `PredictionView` dentro de `last_state` | La respuesta cruda no; su vista sí, vía `last_state` | Sí | Pertenece a una persona: nunca en `cache_resource` (se serviría a otro usuario) ni en la telemetría. |

**Contrato de sesión — ciclo de vida.** `initialize_session_state()` se llama en cada rerun y usa `setdefault`: crea las tres claves solo la primera vez (idempotente). `clear_last_state()` reescribe `last_state` y `last_values` y deja `telemetry` intacta.

**Qué NO se guarda en `telemetry`:** valores de las 11 features, el payload completo, el texto de las excepciones (puede contener rutas o valores), secretos. Solo agregados: `requests_total/success/error`, `errors_by_code`, latencia media y p95, versiones de modelo y preprocesado.

Referencias: [`st.session_state`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state) y [`st.cache_resource`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.cache_resource).

## 3. Diseñar la pantalla, no solo el estado

La interfaz debe tener cuatro zonas, en este orden:

1. **acción:** el formulario heredado de S5;
2. **operación:** qué está ocurriendo mientras se ejecuta;
3. **resultado:** categoría, confianza y latencia;
4. **trazabilidad:** versiones y `request_id`, visibles solo si se necesitan.

Elegid componentes que tengan una función. `st.columns` organiza, `st.status` comunica una operación, `st.metric` destaca un dato y `st.expander` aplaza el detalle técnico. No uséis una barra de progreso para la confianza: representa avance de una tarea, no certeza del modelo.

In [2]:
# Sustituid cada TODO por el componente y la justificación que usaríais.
# Las claves describen zonas de la interfaz; no son nuevos contratos de ML.

ui_plan = {
    'formulario': {
        'component': 'st.form',
        'reason': 'Heredado de S5: agrupa los 11 campos y solo lanza la inferencia al pulsar el botón.',
    },
    'estado_operacion': {
        'component': 'st.status',
        'reason': 'Comunica la operación (en curso → completada/necesita atención) sin desplazar el resultado.',
    },
    'resumen_resultado': {
        'component': 'st.metric',
        'reason': 'Tres métricas en st.columns: categoría, confianza y latencia se leen de un vistazo.',
    },
    'mensaje_interpretacion': {
        'component': 'st.warning',
        'reason': 'Avisa de confianza baja o latencia por encima del objetivo; con confianza media/alta se usa st.info.',
    },
    'trazabilidad': {
        'component': 'st.expander',
        'reason': 'request_id y versiones quedan disponibles para soporte sin invadir el resultado.',
    },
    'separacion_telemetria': {
        'component': 'st.divider',
        'reason': 'Separa la telemetría agregada de la sesión del resultado de la petición actual.',
    },
}

allowed_components = {
    'st.form', 'st.status', 'st.empty', 'st.columns', 'st.container',
    'st.metric', 'st.success', 'st.info', 'st.warning', 'st.error',
    'st.expander', 'st.divider', 'st.caption',
}
required_zones = {
    'formulario', 'estado_operacion', 'resumen_resultado',
    'mensaje_interpretacion', 'trazabilidad', 'separacion_telemetria',
}

assert set(ui_plan) == required_zones
for zone, decision in ui_plan.items():
    assert set(decision) == {'component', 'reason'}, zone

pending = [zone for zone, decision in ui_plan.items() if 'TODO' in decision.values()]
invalid = [
    f"{zone}: {decision['component']}"
    for zone, decision in ui_plan.items()
    if 'TODO' not in decision['component'] and decision['component'] not in allowed_components
]
print('Zonas pendientes:', pending or 'ninguna')
print('Componentes no permitidos:', invalid or 'ninguno')
print('Comprobación final: pending == [] and invalid == []')
assert pending == [] and invalid == []

Zonas pendientes: ninguna
Componentes no permitidos: ninguno
Comprobación final: pending == [] and invalid == []


### Plano mínimo esperado

```text
Título y contexto breve

[ formulario de S5 en dos columnas ]

[ estado de la operación ]

[ Categoría ]  [ Confianza ]  [ Latencia ]
mensaje interpretativo de confianza y latencia
▶ Trazabilidad: versiones y request_id

────────────────────────────────────
Telemetría agregada de la sesión
```

Podéis entregar este plano en el notebook o redibujarlo en draw.io. Justificad cualquier cambio respecto a esta estructura.

**Decisiones de la pareja respecto al plano:**

- La zona de resultado es un `st.empty()` reservado: `loading`, `success` y `error` se dibujan en el mismo hueco, así que la pantalla no «salta».
- Los botones «Reintentar» y «Limpiar último resultado» van justo debajo del resultado; «Reintentar» solo se habilita en `error` recuperable.
- **Cambio en el [laboratorio visual](ui_playground.py):** se elimina el banner `st.success(quality_label)` del escenario de éxito. Duplicaba la categoría y pintaba de verde también «Requiere revisión», lo que comunica «el modelo acierta». El verde (éxito) es de la *operación* (`st.status` completado), no de la predicción; la categoría queda en una única `st.metric` con la etiqueta «Categoría».

## 4. Máquina de estados y recuperación

Completad la tabla. Una respuesta válida con latencia superior a 300 ms sigue en `success`: añade un aviso técnico, no se convierte en error. La confianza es una señal para revisar, no un diagnóstico ni una garantía.

| Estado | Evento de entrada | Zona de pantalla que cambia | Qué se muestra | Acción permitida | Evidencia |
| --- | --- | --- | --- | --- | --- |
| `idle` | Inicio de sesión / pulsar «Limpiar» | Zona de resultado (`st.empty`) | `st.info` con la instrucción; sin métricas vacías ni detalle técnico | Rellenar y enviar el formulario | `last_state == UiState("idle")`; tras limpiar, `telemetry.requests_total` no cambia |
| `loading` | Enviar formulario / pulsar «Reintentar» | Operación (`st.status` en curso) y zona de resultado | «Ejecutando inferencia…» y aviso de que el resultado aparecerá en esa zona; el formulario sigue visible | Esperar | El controlador emite `UiState("loading", request_id)` **antes** de llamar al gateway |
| `success` | El gateway devuelve un payload que pasa `PredictionPayload` | `st.status` → completada; zona de resultado | 3 `st.metric` (categoría, confianza, latencia); mensaje de confianza (`st.warning` si es baja, `st.info` si no); `st.warning` si latencia > 300 ms; expander «Trazabilidad» | Nuevo envío, «Limpiar» | `requests_success` +1 y versiones registradas; el resultado sigue visible tras un rerun |
| `error` | Excepción controlada (contrato de entrada, bundle, timeout, salida fuera de contrato) | `st.status` → necesita atención; zona de resultado | `st.error` con título y mensaje estable, `st.info` con la recuperación y el `request_id` | «Reintentar» (salvo `invalid_input`), corregir y reenviar, «Limpiar» | `errors_by_code[code]` +1; sin traceback, rutas ni valores |

**Recuperación por caso:**

| Caso | Código | Qué ve la persona | Recuperación |
| --- | --- | --- | --- |
| Entrada inválida | `invalid_input` | «Datos de entrada no válidos» | Corregir el formulario y reenviar. «Reintentar» queda deshabilitado: los mismos valores fallarían igual. |
| Bundle ausente o que no carga | `artifact_unavailable` | «Modelo no disponible» | «Reintentar» invalida `st.cache_resource` y vuelve a cargar el gateway (no se cachea un fallo para siempre); si persiste, comunicar el `request_id`. |
| Fallo transitorio / timeout | `timeout` | «La inferencia tardó demasiado» | «Reintentar» reenvía `last_values`. |
| Salida fuera de contrato | `prediction_error` | «No se pudo obtener una predicción válida» | «Reintentar»; si persiste, comunicar el `request_id`. |
| Limpiar | — | Vuelve a `idle` | Borra `last_state` y `last_values`; **conserva** `telemetry`. |

**Lo que la persona nunca debe ver:** traceback, texto original de la excepción, rutas internas (p. ej. la del `.joblib`), secretos, el payload completo o los valores de las features. El detalle técnico queda en el log del servidor, correlacionado por `request_id` (solo clase de excepción y código).

## 5. Escribir el copy de una interfaz responsable

Cada mensaje debe decir qué ha ocurrido, qué significa y qué puede hacer la persona. No debéis prometer que una predicción es correcta. Completad textos de una frase, sin terminología interna.

In [3]:
ux_copy = {
    'low_confidence': (
        'Confianza baja: el resultado es orientativo y requiere revisión; '
        'la confianza no es una garantía.'
    ),
    'slow_success': (
        'La respuesta es válida, pero tardó más que el objetivo técnico de la demo '
        '(300 ms); no indica un fallo del modelo.'
    ),
    'invalid_input': (
        'Algunos valores del formulario no cumplen el contrato de entrada; '
        'revísalos y vuelve a enviarlo.'
    ),
    'artifact_unavailable': (
        'El modelo no está disponible ahora mismo y no se ha hecho ninguna predicción; '
        'pulsa «Reintentar» más tarde o comunica el request_id.'
    ),
}

assert set(ux_copy) == {
    'low_confidence', 'slow_success', 'invalid_input', 'artifact_unavailable'
}
for scenario, message in ux_copy.items():
    assert isinstance(message, str) and message.strip(), scenario

pending_copy = [scenario for scenario, message in ux_copy.items() if 'TODO' in message]
print('Mensajes pendientes:', pending_copy or 'ninguno')
print('Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?')
assert pending_copy == []

Mensajes pendientes: ninguno
Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?


## 6. Preparar el taller

Entregad el notebook con los TODO completados. Antes de cerrar, otra pareja debe poder responder estas preguntas leyendo vuestro trabajo:

- ¿Qué se mantiene al pulsar otro widget y qué se elimina con «Limpiar último resultado»?
- ¿Qué función cargará el gateway una sola vez y cuál será su límite?
- ¿Qué componentes dibujará `render_state()` para `loading`, `success` y `error`?
- ¿Qué ve una persona con confianza baja o una respuesta lenta?
- ¿Dónde aparecen el `request_id` y las versiones sin invadir el resultado?

En el taller se implementarán estas decisiones en `session.py`, `policies.py`, `presentation.py`, `controller.py` y `app.py`.

**Respuestas de la pareja:**

- **Otro widget:** se conservan `last_state` (el resultado sigue visible), `last_values` y `telemetry`. «Limpiar último resultado» elimina `last_state` y `last_values`, nunca `telemetry`.
- **Carga única:** `get_cached_gateway()` envuelve `build_gateway()` con `st.cache_resource`. Límite: se comparte entre sesiones, así que solo puede contener el bundle de solo lectura, y un fallo de carga quedaría cacheado (por eso «Reintentar» tras `artifact_unavailable` invalida la caché).
- **`render_state()`:** `loading` → `st.info` en la zona reservada + `st.status` en curso; `success` → 3 `st.metric`, `st.warning`/`st.info` y `st.expander`; `error` → `st.error` + `st.info` con recuperación + `request_id`.
- **Confianza baja / respuesta lenta:** `st.warning` «Confianza baja… no es una garantía» y `st.warning` «Por encima del objetivo… Objetivo: 300 ms». La respuesta lenta sigue en `success`.
- **`request_id` y versiones:** en el expander «Trazabilidad» del éxito; en error, el `request_id` aparece como referencia bajo la recuperación.

## 7. Decisión de caché

| Recurso cacheable | Argumento de caché | Dato excluido |
| --- | --- | --- |
| El gateway (`DemoGateway` o `PackagedBundleGateway` con el bundle de S4 cargado), construido por `build_gateway()` | `bundle_value` (ruta de `MODEL_UI_BUNDLE`) y, sin bundle, el escenario docente; si cambian, se crea otro recurso | Valores del formulario, `PredictionPayload`/`UiState` de una persona y cualquier secreto |

`st.cache_resource` devuelve **el mismo objeto** a todas las sesiones del proceso. Si se cacheara una respuesta, una persona podría ver la predicción de otra; por eso la respuesta vive en `st.session_state` y el recurso cacheado no guarda estado por petición (su `predict` es de solo lectura). `st.cache_data` tampoco se usa para las predicciones: el objetivo es medir y comunicar cada inferencia real, con su `request_id` y su latencia.

## 8. Plan de implementación (práctica 6.2)

| Requisito del diseño | Archivo y función | Test del starter |
| --- | --- | --- |
| Claves creadas una vez, sin pisar la telemetría | `session.py` · `initialize_session_state` | `test_session_state_initialization_is_idempotent` |
| Limpiar vuelve a `idle` y conserva la telemetría | `session.py` · `clear_last_state` | `test_clear_last_state_keeps_telemetry` |
| Confianza: `< 0.60` low, `>= 0.85` high, fuera de `[0, 1]` error | `policies.py` · `classify_confidence` | `test_confidence_policy_has_explicit_boundary_values` |
| Latencia: `<= 300 ms` within_target | `policies.py` · `classify_latency` | `test_latency_and_confidence_are_explained_to_the_user` |
| View model con copy seguro, versiones y mensaje de latencia | `presentation.py` · `build_prediction_view` | `test_latency_and_confidence_are_explained_to_the_user` |
| Errores con código estable, sin detalle técnico | `presentation.py` · `to_user_facing_error` | `test_error_translation_does_not_expose_technical_detail` |
| `loading → success`, latencia medida, versiones registradas | `controller.py` · `PredictionController.submit` | `test_controller_emits_loading_then_success_and_records_versions` |
| `loading → error`, telemetría sin payload | `controller.py` · `PredictionController.submit` | `test_controller_emits_loading_then_error_without_payload_telemetry` |
| Contrato de entrada de S5 (no se toca) | `gateway.py` · `_validate_feature_names` | `test_demo_gateway_rejects_unknown_and_missing_features` |
| Gateway cargado una vez | `app.py` · `get_cached_gateway` | Revisión visual: el resultado y el gateway sobreviven a reruns |
| Estados visibles, retry y clear | `app.py` · `render_state`, `run_prediction`, `main` | Revisión visual de los recorridos: inicio, éxito, rerun, error, reintentar, limpiar |

Orden: sesión → políticas → presentación → controlador → app (cada paso pone en verde sus tests antes de seguir).

## Debrief

- ¿Qué limita a S5 aunque el formulario ya funcione?
- ¿Por qué el gateway es un recurso cacheable y el resultado de una petición pertenece a una sesión?
- ¿Qué componente hace visible cada transición sin añadir una segunda lógica de inferencia?
- ¿Por qué categoría, confianza y latencia no tienen el mismo significado?
- ¿Qué parte de este diseño se conserva al sustituir el gateway local por HTTP en S7?